In [ ]:
"""
Approval Engine Module

This module manages approval workflows for content moderation including
submission, response handling, timeout checking, and status tracking.
"""

from typing import Dict, List, Optional, Any
from datetime import datetime, timedelta
from enum import Enum
import uuid


class ApprovalStatus(Enum):
    """Approval request status"""

    PENDING = "pending"
    APPROVED = "approved"
    REJECTED = "rejected"
    MODIFIED = "modified"
    EXPIRED = "expired"


class PendingAction:
    """Represents a pending action awaiting approval"""

    def __init__(
        self,
        action_id: str,
        action_type: str,
        payload: dict,
        created_at: datetime,
        expires_at: datetime,
        required_approvers: List[str] = None,
    ):
        self.action_id = action_id
        self.action_type = action_type
        self.payload = payload
        self.created_at = created_at
        self.expires_at = expires_at
        self.status = ApprovalStatus.PENDING
        self.required_approvers = required_approvers or []
        self.approvals_received: List[Dict] = []
        self.modifications: Optional[Dict] = None


class ApprovalEngine:
    """
    Manages approval workflows for content moderation.
    """

    def __init__(self, default_timeout_hours: int = 24):
        """Initialize approval engine"""
        self.default_timeout_hours = default_timeout_hours
        self._pending_actions: Dict[str, PendingAction] = {}

    def submit_for_approval(
        self,
        action_type: str,
        payload: dict,
        required_approvers: List[str] = None,
        timeout_hours: Optional[int] = None,
    ) -> PendingAction:
        """
        Submit an action for approval.
        """
        action_id = str(uuid.uuid4())
        created_at = datetime.utcnow()

        expires_at = created_at + timedelta(
            hours=timeout_hours or self.default_timeout_hours
        )
        approvers = required_approvers or []

        pending = PendingAction(
            action_id=action_id,
            action_type=action_type,
            payload=payload,
            created_at=created_at,
            expires_at=expires_at,
            required_approvers=approvers,
        )

        self._pending_actions[action_id] = pending
        return pending

    def process_approval_response(
        self,
        action_id: str,
        approver: str,
        decision: str,
        modifications: Optional[Dict] = None,
    ) -> Dict:
        """
        Process an approval response.

        Supported decisions:
            - approved
            - rejected
            - modified

        Returns an error for unknown decisions.
        """
        if action_id not in self._pending_actions:
            return {"error": f"Action {action_id} not found"}

        pending = self._pending_actions[action_id]

        # Normalize decision while preserving the public API values.
        normalized_decision = decision.lower()

        if normalized_decision not in {
            "approved",
            "rejected",
            "modified",
        }:
            return {
                "error": f"Invalid decision: {decision}"
            }

        # Map the decision to the enum.
        pending.status = ApprovalStatus(normalized_decision)

        # Apply modifications to the actual payload.
        if modifications:
            pending.modifications = modifications
            payload_updates = modifications.get("payload_updates", modifications)

            if isinstance(payload_updates, dict):
                pending.payload.update(payload_updates)

        # Every approver response must be preserved.
        pending.approvals_received.append(
            {
                "approver": approver,
                "decision": normalized_decision,
                "timestamp": datetime.utcnow(),
            }
        )

        return {
            "action_id": action_id,
            "status": pending.status.value,
            "modifications": pending.modifications,
        }

    def check_timeouts(self) -> List[Dict]:
        """
        Check for expired approvals and handle them.
        """

        current_time = datetime.now()
        expired_actions = []

        for action in self._pending_actions.values():
            if (
                action.status == ApprovalStatus.PENDING
                and current_time > action.expires_at
            ):
                action.status = ApprovalStatus.EXPIRED
                expired_actions.append(
                    {
                        "action_id": action.action_id,
                        "action_type": action.action_type,
                        "expired_at": action.expires_at,
                        "status": action.status.value,
                    }
                )

        return expired_actions

    def get_pending_approvals(self, approver: str) -> List[PendingAction]:
        """
        Get all pending approvals for a specific approver.

        Args:
            approver: User ID of the approver

        Returns:
            List of PendingAction objects requiring this approver's approval
        """
        pending = []
        for action in self._pending_actions.values():
            if action.status == ApprovalStatus.PENDING:
                if approver in action.required_approvers:
                    pending.append(action)
        return pending

    def get_action(self, action_id: str) -> Optional[PendingAction]:
        """Get a pending action by ID"""
        return self._pending_actions.get(action_id)

    def clear(self):
        """Clear all pending actions (for testing)"""
        self._pending_actions.clear()


In [ ]:
"""
Feedback Collector Module

This module collects and tracks feedback for moderation actions including
explicit feedback (user ratings) and implicit feedback (behavioral signals).
"""

from typing import Dict, List, Optional
from datetime import datetime
from enum import Enum
import uuid


class FeedbackType(Enum):
    """Types of explicit feedback"""
    HELPFUL = "helpful"
    NOT_HELPFUL = "not_helpful"
    INCORRECT = "incorrect"
    GOOD_CATCH = "good_catch"


class FeedbackItem:
    """Represents a single feedback item"""

    def __init__(self, feedback_id: str, action_id: str, feedback_type: str,
                 user: str, timestamp: datetime, text_feedback: Optional[str] = None):
        self.feedback_id = feedback_id
        self.action_id = action_id
        self.feedback_type = feedback_type
        self.user = user
        self.timestamp = timestamp
        self.text_feedback = text_feedback


class FeedbackCollector:
    """
    Collects and tracks feedback for moderation actions.
    """

    def __init__(self):
        """Initialize feedback collector"""
        self._explicit_feedback: List[FeedbackItem] = []
        self._implicit_feedback: List[Dict] = []

    def record_feedback(self, action_id: str, feedback_type: str, user: str,
                       text_feedback: Optional[str] = None) -> FeedbackItem:
        """
        Record explicit feedback (helpful, not_helpful, incorrect, good_catch).
        """
        feedback_id = str(uuid.uuid4())
        timestamp = datetime.utcnow()

        feedback_item = FeedbackItem(
            feedback_id=feedback_id,
            action_id=action_id,
            feedback_type=feedback_type,
            user=user,
            timestamp=timestamp,
            text_feedback=text_feedback
        )

        self._explicit_feedback.append(feedback_item)
        return feedback_item

    def record_implicit_feedback(self, action_id: str, signal: str,
                                metadata: Optional[Dict] = None) -> Dict:
        """
        Record implicit feedback signal (content_accepted, content_rejected, content_modified).
        """
        feedback_id = str(uuid.uuid4())
        timestamp = datetime.utcnow().isoformat()

        feedback_record = {
            "feedback_id": feedback_id,
            "action_id": action_id,
            "signal": signal,
            "metadata": metadata or {},
            "timestamp": timestamp
        }

        self._implicit_feedback.append(feedback_record)
        return feedback_record

    def get_feedback_summary(self, action_id: str) -> Dict:
        """
        Get summary of feedback for a specific action.
        """
        explicit_count = 0
        implicit_signals = []

        implicit_count = 0
        feedback_types = {}

        for feedback in self._explicit_feedback:
            if feedback.action_id == action_id:
                explicit_count += 1
                feedback_types[feedback.feedback_type] = feedback_types.get(feedback.feedback_type, 0) + 1

        for feedback in self._implicit_feedback:
            if feedback["action_id"] == action_id:
                implicit_count += 1
                implicit_signals.append(feedback["signal"])

        return {
            "action_id": action_id,
            "explicit_count": explicit_count,
            "implicit_count": implicit_count,
            "feedback_types": feedback_types,
            "implicit_signals": implicit_signals
        }

    def get_satisfaction_score(self, action_id: Optional[str] = None) -> float:
        """
        Calculate satisfaction score from feedback.
        """
        explicit_weights = {
            FeedbackType.HELPFUL.value: 1.0,
            FeedbackType.GOOD_CATCH.value: 0.8,
            FeedbackType.NOT_HELPFUL.value: -0.5,
            FeedbackType.INCORRECT.value: -1.0
        }

        implicit_weights = {
            "content_accepted": 0.5,
            "content_rejected": -0.5,
            "content_modified": 0.0
        }

        total_score = 0.0
        count = 0

        for feedback in self._explicit_feedback:
            if action_id is None or feedback.action_id == action_id:
                weight = explicit_weights.get(feedback.feedback_type, 0.0)
                total_score += weight
                count += 1

        for feedback in self._implicit_feedback:
            if action_id is None or feedback["action_id"] == action_id:
                weight = implicit_weights.get(feedback["signal"], 0.0)
                total_score += weight
                count += 1

        return total_score / count if count > 0 else 0.0

    def clear(self):
        """Clear all feedback (for testing)"""
        self._explicit_feedback.clear()
        self._implicit_feedback.clear()


In [ ]:
"""
Content Moderation Agent

This module implements a content moderation agent with human-in-the-loop (HITL)
patterns including approval workflows and feedback collection.
"""

from typing import Dict, List, Optional, Any
import json
from datetime import datetime
import uuid

from approval_engine import ApprovalEngine, ApprovalStatus
from feedback_collector import FeedbackCollector
from llm import get_llm_client


class ContentModerationAgent:
    """
    Content moderation agent with HITL patterns.
    """

    def __init__(self, default_approvers: List[str] = None):
        """
        Initialize content moderation agent.

        Args:
            default_approvers: Default list of approvers for high-risk content
        """
        self.approval_engine = ApprovalEngine(default_timeout_hours=24)
        self.feedback_collector = FeedbackCollector()
        self.llm = get_llm_client()
        self.default_approvers = default_approvers or ["moderator_1", "moderator_2"]

    def should_require_approval(self, risk_level: str, confidence: float) -> bool:
        """
        Determine if content requires human approval.
        """
        if risk_level == "critical":
            return True
        elif risk_level == "high" or confidence < 0.7:
            return True
        elif risk_level == "low" and confidence >= 0.8:
            return False
        elif risk_level == "medium" and confidence >= 0.8:
            return False
        return True

    async def moderate_content(self, content: str, content_id: str,
                              user_id: str) -> Dict:
        """
        Moderate user-generated content.
        """
        action_id = f"action_{content_id}"

        analysis = await self._analyze_content(content)

        risk_level = analysis["risk_level"]
        confidence = analysis["confidence"]

        require_approval = self.should_require_approval(
            risk_level,
            confidence
        )

        if require_approval:
            approval_request = self._create_approval_request(
                content,
                content_id,
                risk_level,
                confidence,
                analysis,
            )

            pending_action = self.approval_engine.submit_for_approval(
                action_type="content_moderation",
                payload=approval_request,
                required_approvers=self.default_approvers,
            )

            return {
                "action_id": action_id,
                "decision": "pending_approval",
                "risk_level": risk_level,
                "confidence": confidence,
                "requires_approval": True,
                "approval_id": pending_action.action_id,
            }

        decision = (
            "approved"
            if risk_level in ("low", "medium")
            else "rejected"
        )

        self._collect_feedback(
            action_id,
            decision,
            user_id,
        )

        return {
            "action_id": action_id,
            "decision": decision,
            "risk_level": risk_level,
            "confidence": confidence,
            "requires_approval": False,
        }

    def _create_approval_request(self, content: str, content_id: str,
                                risk_level: str, confidence: float,
                                analysis: Dict) -> Dict:
        """
        Create an approval request with full context.
        """
        content_summary = content[:500] + ("..." if len(content) > 500 else "")
        risk_assessment = {
            "risk_level": risk_level,
            "confidence": confidence,
            "reasoning": analysis.get("reasoning", "")
        }
        proposed_action = "reject" if risk_level in ("high", "critical") else "approve"
        key_findings = [risk_level, confidence, analysis.get("reasoning", "")]
        approval_effect = {"action": proposed_action, "content_id": content_id}
        rejection_effect = {"action": "approve", "content_id": content_id} if proposed_action == "reject" else None

        return {
            "content_id": content_id,
            "content": content,
            "content_summary": content_summary,
            "risk_assessment": risk_assessment,
            "proposed_action": proposed_action,
            "key_findings": key_findings,
            "approval_effect": approval_effect,
            "rejection_effect": rejection_effect
        }

    def _collect_feedback(self, action_id: str, decision: str, user_id: str):
        """
        Collect feedback after moderation action.
        """
        if decision == "approved":
            self.feedback_collector.record_implicit_feedback(
                action_id, "content_accepted", {"user_id": user_id}
            )
        elif decision == "rejected":
            self.feedback_collector.record_implicit_feedback(
                action_id, "content_rejected", {"user_id": user_id}
            )
        elif decision == "modified":
            self.feedback_collector.record_implicit_feedback(
                action_id, "content_modified", {"user_id": user_id}
            )

    async def _analyze_content(self, content: str) -> Dict:
        """
        Analyze content using LLM to determine risk and confidence.

        Args:
            content: Content to analyze

        Returns:
            Dictionary with risk_level, confidence, and analysis
        """
        messages = [
            {
                "role": "system",
                "content": "You are a content moderation assistant. Analyze content and determine risk level (low, medium, high, critical) and your confidence (0.0 to 1.0). Return JSON with 'risk_level', 'confidence', and 'reasoning'."
            },
            {
                "role": "user",
                "content": f"Analyze this content for policy violations:\n\n{content}"
            }
        ]

        response = self.llm.chat_completion(messages, model="gpt-4o-mini", max_tokens=200)

        try:
            content_text = response.choices[0].message.content
            analysis = json.loads(content_text)
            return {
                "risk_level": analysis.get("risk_level", "medium"),
                "confidence": float(analysis.get("confidence", 0.5)),
                "reasoning": analysis.get("reasoning", "")
            }
        except (json.JSONDecodeError, KeyError, ValueError):
            # Fallback if LLM doesn't return valid JSON
            return {
                "risk_level": "medium",
                "confidence": 0.5,
                "reasoning": "Unable to parse LLM response"
            }
